In [76]:
import pandas as pd

In [77]:
df_head = pd.read_csv('yellow_tripdata_2021-01.csv.gz', nrows=100)

In [78]:
print("--- First 100 Rows ---")
print(df_head.head())

--- First 100 Rows ---
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_a

In [79]:
print("\n--- Data Types ---")
print(df_head.dtypes)


--- Data Types ---
VendorID                   int64
tpep_pickup_datetime      object
tpep_dropoff_datetime     object
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag        object
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object


In [80]:
total_rows = 0
null_counts = None
min_values = {}
max_values = {}
dtypes = None

In [81]:
chunk_size = 100000
chunks = pd.read_csv('yellow_tripdata_2021-01.csv.gz', chunksize=chunk_size, low_memory=False)

total_rows = 0
null_counts = None
min_values = {}
max_values = {}
dtypes = None

for chunk in chunks:
    total_rows += len(chunk)
    
    if dtypes is None:
        dtypes = chunk.dtypes
        
    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()
        
    for col in chunk.columns:
        try:
            c_min = chunk[col].min()
            c_max = chunk[col].max()
            if col not in min_values or c_min < min_values[col]:
                min_values[col] = c_min
            if col not in max_values or c_max > max_values[col]:
                max_values[col] = c_max
        except Exception:
            pass

In [82]:
profile_df = pd.DataFrame({
    'dtype': dtypes,
    'null_count': null_counts,
    'null_pct': (null_counts / total_rows) * 100,
    'min': pd.Series(min_values),
    'max': pd.Series(max_values)
})

print("--- Profiling Results ---")
print(profile_df)

--- Profiling Results ---
                         dtype  null_count  null_pct                  min  \
VendorID                 int64       98352   7.18021                    1   
tpep_pickup_datetime    object           0   0.00000  2008-12-31 23:05:14   
tpep_dropoff_datetime   object           0   0.00000  2008-12-31 23:07:22   
passenger_count          int64       98352   7.18021                    0   
trip_distance          float64           0   0.00000                  0.0   
RatecodeID               int64       98352   7.18021                    1   
store_and_fwd_flag      object       98352   7.18021                    N   
PULocationID             int64           0   0.00000                    1   
DOLocationID             int64           0   0.00000                    1   
payment_type             int64       98352   7.18021                    1   
fare_amount            float64           0   0.00000               -490.0   
extra                  float64           0   0.000

In [83]:
def clean_chunk(chunk_df):
    cleaned = chunk_df.copy()
    removed_rules = {}

    
    cleaned['tpep_pickup_datetime'] = pd.to_datetime(cleaned['tpep_pickup_datetime'])
    
    
    mask_date = (cleaned['tpep_pickup_datetime'] >= '2021-01-01') & (cleaned['tpep_pickup_datetime'] < '2021-02-01')
    removed_rules['out_of_range_dates'] = len(cleaned) - mask_date.sum()
    cleaned = cleaned[mask_date]

    
    mask_valid = (cleaned['trip_distance'] > 0) & (cleaned['fare_amount'] > 0)
    removed_rules['invalid_fare_or_distance'] = len(cleaned) - mask_valid.sum()
    cleaned = cleaned[mask_valid]

    
    mask_passengers = (cleaned['passenger_count'] > 0) | (cleaned['passenger_count'].isna())
    removed_rules['zero_passengers'] = len(cleaned) - mask_passengers.sum()
    cleaned = cleaned[mask_passengers]

    
    before_dup = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    removed_rules['duplicate_rows'] = before_dup - len(cleaned)

    return cleaned, removed_rules

In [84]:
total_rows_before = 0
total_rows_after = 0
total_removed_per_rule = {
    'out_of_range_dates': 0,
    'invalid_fare_or_distance': 0,
    'zero_passengers': 0,
    'duplicate_rows': 0
}

chunks = pd.read_csv('yellow_tripdata_2021-01.csv.gz', chunksize=100000, low_memory=False)

for chunk in chunks:
    total_rows_before += len(chunk)
    
    
    cleaned_c, removed_c = clean_chunk(chunk)
    
    total_rows_after += len(cleaned_c)
    
    
    for rule, count in removed_c.items():
        total_removed_per_rule[rule] += count


print(f"Total Rows Before: {total_rows_before}")
print(f"Total Rows After:  {total_rows_after}")
print("\n--- Rows Removed Per Rule ---")
print(pd.Series(total_removed_per_rule))

Total Rows Before: 1369765
Total Rows After:  1317117

--- Rows Removed Per Rule ---
out_of_range_dates             24
invalid_fare_or_distance    26378
zero_passengers             26246
duplicate_rows                  0
dtype: int64
